<a href="https://colab.research.google.com/github/May-ysaa/InvoiceFlow-AI/blob/main/notebooks/00_environment_setup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


**Cell 0 — Markdown: Judul notebook**

# InvoiceFlow AI — Environment Setup

**Notebook ID:** `00_environment_setup.ipynb`  
**Project:** InvoiceFlow AI – Bulk Invoice Intelligence  
**Environment:** Google Colab  
**Repository:** `May-ysaa/InvoiceFlow-AI`  
**Stage:** Development Environment Initialization  

## Purpose

Notebook ini menyiapkan lingkungan pengembangan InvoiceFlow AI secara aman,
konsisten, dan dapat direproduksi.

Proses yang dilakukan:

1. memvalidasi runtime Python;
2. menghubungkan Google Drive;
3. menghubungkan repository GitHub;
4. membuat struktur proyek;
5. memasang dan memverifikasi dependensi;
6. memisahkan source code dan data privat;
7. mengatur reproducibility dan logging;
8. melakukan security check;
9. menyimpan environment manifest.

> Invoice asli dan data privat tidak boleh disimpan di repository GitHub.

**Cell 1 — Markdown: Tujuan**

## 1. Objective

Environment dinyatakan siap hanya jika:

- versi Python didukung;
- repository GitHub dapat diakses;
- Google Drive dapat ditulis;
- seluruh dependensi utama berhasil diimpor;
- PaddlePaddle berhasil menjalankan pemeriksaan internal;
- struktur folder kode dan data tersedia;
- tidak ditemukan file rahasia di repository;
- konfigurasi lingkungan berhasil dicatat.

**Cell 2 — Code: Pemeriksaan runtime awal**

In [ ]:
from __future__ import annotations

import importlib.util
import os
import platform
import shutil
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path


SUPPORTED_PYTHON_MIN = (3, 10)
SUPPORTED_PYTHON_MAX = (3, 13)

CURRENT_PYTHON = sys.version_info[:2]
IN_COLAB = importlib.util.find_spec("google.colab") is not None
IS_64_BIT = sys.maxsize > 2**32

if not SUPPORTED_PYTHON_MIN <= CURRENT_PYTHON <= SUPPORTED_PYTHON_MAX:
    raise RuntimeError(
        "Versi Python tidak didukung. "
        f"Ditemukan {CURRENT_PYTHON}, diperlukan "
        f"{SUPPORTED_PYTHON_MIN}–{SUPPORTED_PYTHON_MAX}."
    )

if not IS_64_BIT:
    raise RuntimeError("InvoiceFlow AI membutuhkan Python 64-bit.")

runtime_information = {
    "python_version": sys.version.split()[0],
    "python_executable": sys.executable,
    "operating_system": platform.system(),
    "platform": platform.platform(),
    "architecture": platform.machine(),
    "is_64_bit": IS_64_BIT,
    "is_google_colab": IN_COLAB,
}

for key, value in runtime_information.items():
    print(f"{key:20}: {value}")

print("\n✅ Pemeriksaan runtime berhasil.")

**Cell 3 — Code: Konfigurasi pusat proyek**

In [ ]:
PROJECT_NAME = "InvoiceFlow-AI"
PROJECT_VERSION = "0.1.0"
ENVIRONMENT_NAME = "development"

GITHUB_OWNER = "May-ysaa"
GITHUB_REPOSITORY = "InvoiceFlow-AI"
DEFAULT_BRANCH = "main"

GITHUB_URL = (
    f"https://github.com/{GITHUB_OWNER}/{GITHUB_REPOSITORY}.git"
)

LOCAL_REPOSITORY_ROOT = Path("/content") / GITHUB_REPOSITORY
DRIVE_MOUNT_POINT = Path("/content/drive")
DATA_ROOT = (
    DRIVE_MOUNT_POINT
    / "MyDrive"
    / "InvoiceFlow-AI-Data"
)

RANDOM_SEED = 42
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

print(f"Project     : {PROJECT_NAME}")
print(f"Version     : {PROJECT_VERSION}")
print(f"Environment : {ENVIRONMENT_NAME}")
print(f"Repository  : {GITHUB_URL}")
print(f"Code root   : {LOCAL_REPOSITORY_ROOT}")
print(f"Data root   : {DATA_ROOT}")
print(f"Run ID      : {RUN_ID}")

**Cell 4 — Markdown: Kebijakan penyimpanan**

### Storage Policy

| Jenis data | Lokasi | GitHub |
|---|---|---|
| Source code | `/content/InvoiceFlow-AI` | Diizinkan |
| Notebook | Repository `notebooks/` | Diizinkan |
| Unit test | Repository `tests/` | Diizinkan |
| Sampel sintetis | Repository atau Drive | Diizinkan jika aman |
| Invoice asli | Google Drive `raw/` | Dilarang |
| Hasil OCR sementara | Google Drive `interim/` | Dilarang |
| Data terstruktur privat | Google Drive `processed/` | Dilarang |
| Token dan secret | Colab Secrets/environment | Dilarang |

**Cell 5 — Code: Menghubungkan Google Drive**

In [ ]:
if not IN_COLAB:
    raise RuntimeError(
        "Notebook ini dirancang untuk Google Colab. "
        "Buka notebook menggunakan Google Colab."
    )

from google.colab import drive

drive.mount(
    str(DRIVE_MOUNT_POINT),
    force_remount=False,
)

if not DRIVE_MOUNT_POINT.exists():
    raise RuntimeError("Google Drive gagal dihubungkan.")

print("✅ Google Drive berhasil dihubungkan.")

**Cell 6 — Code: Fungsi menjalankan perintah sistem**

In [ ]:
def run_command(
    command: list[str],
    working_directory: Path | None = None,
    check: bool = True,
) -> subprocess.CompletedProcess:
    """
    Menjalankan command dengan error handling yang jelas.
    """

    result = subprocess.run(
        command,
        cwd=str(working_directory) if working_directory else None,
        capture_output=True,
        text=True,
        check=False,
    )

    if result.stdout.strip():
        print(result.stdout.strip())

    if result.returncode != 0 and check:
        error_message = result.stderr.strip() or "Unknown command error"
        raise RuntimeError(
            f"Command gagal: {' '.join(command)}\n"
            f"Detail: {error_message[-4000:]}"
        )

    return result


git_version_result = run_command(["git", "--version"])
print("✅ Git tersedia pada runtime.")

**Cell 7 — Code: Clone atau sinkronisasi repository**

In [ ]:
git_directory = LOCAL_REPOSITORY_ROOT / ".git"

if git_directory.exists():
    print("Repository sudah tersedia. Melakukan sinkronisasi aman...")

    run_command(
        ["git", "fetch", "origin", "--prune"],
        working_directory=LOCAL_REPOSITORY_ROOT,
    )

    remote_branch_check = run_command(
        [
            "git",
            "rev-parse",
            "--verify",
            f"origin/{DEFAULT_BRANCH}",
        ],
        working_directory=LOCAL_REPOSITORY_ROOT,
        check=False,
    )

    if remote_branch_check.returncode == 0:
        current_branch = run_command(
            ["git", "branch", "--show-current"],
            working_directory=LOCAL_REPOSITORY_ROOT,
        ).stdout.strip()

        if current_branch == DEFAULT_BRANCH:
            run_command(
                ["git", "pull", "--ff-only", "origin", DEFAULT_BRANCH],
                working_directory=LOCAL_REPOSITORY_ROOT,
            )
        else:
            print(
                f"ℹ️ Branch aktif adalah '{current_branch}'. "
                "Pull otomatis tidak dilakukan."
            )
    else:
        print("ℹ️ Repository GitHub belum mempunyai commit.")
else:
    if (
        LOCAL_REPOSITORY_ROOT.exists()
        and any(LOCAL_REPOSITORY_ROOT.iterdir())
    ):
        raise RuntimeError(
            f"Folder {LOCAL_REPOSITORY_ROOT} sudah berisi file, "
            "tetapi bukan repository Git."
        )

    run_command(
        [
            "git",
            "clone",
            GITHUB_URL,
            str(LOCAL_REPOSITORY_ROOT),
        ]
    )

if not git_directory.exists():
    raise RuntimeError("Repository gagal dihubungkan.")

print(f"✅ Repository aktif: {LOCAL_REPOSITORY_ROOT}")

**Cell 8 — Code: Membuat struktur source code**

In [ ]:
NOTEBOOKS_DIR = LOCAL_REPOSITORY_ROOT / "notebooks"
SOURCE_DIR = LOCAL_REPOSITORY_ROOT / "src"
APPLICATION_DIR = LOCAL_REPOSITORY_ROOT / "app"
TESTS_DIR = LOCAL_REPOSITORY_ROOT / "tests"

CODE_DIRECTORIES = [
    NOTEBOOKS_DIR,
    SOURCE_DIR,
    APPLICATION_DIR,
    TESTS_DIR,
]

for directory in CODE_DIRECTORIES:
    directory.mkdir(parents=True, exist_ok=True)

print("Struktur source code:")

for directory in CODE_DIRECTORIES:
    print(f"  ✅ {directory.relative_to(LOCAL_REPOSITORY_ROOT)}")

**Cell 9 — Code: Membuat requirements.txt**

In [ ]:
REQUIREMENTS_PATH = LOCAL_REPOSITORY_ROOT / "requirements.txt"

DEFAULT_REQUIREMENTS = """
paddlepaddle==3.2.0
paddleocr>=3.0,<4.0
PyMuPDF>=1.24,<2.0
Pillow>=10.0,<13.0
opencv-python-headless>=4.10,<5.0
pandas>=2.2,<4.0
numpy>=1.26,<3.0
scikit-learn>=1.5,<2.0
sentence-transformers>=3.0,<6.0
streamlit>=1.40,<2.0
plotly>=5.24,<7.0
openpyxl>=3.1,<4.0
pytest>=8.0,<10.0
""".strip() + "\n"

if not REQUIREMENTS_PATH.exists():
    REQUIREMENTS_PATH.write_text(
        DEFAULT_REQUIREMENTS,
        encoding="utf-8",
    )
    print("✅ requirements.txt berhasil dibuat.")
else:
    print("ℹ️ requirements.txt sudah ada dan tidak ditimpa.")

print(REQUIREMENTS_PATH.read_text(encoding="utf-8"))

**Cell 10 — Code: Membuat .gitignore**

In [ ]:
GITIGNORE_PATH = LOCAL_REPOSITORY_ROOT / ".gitignore"

REQUIRED_GITIGNORE_RULES = [
    # Python
    "__pycache__/",
    "*.py[cod]",
    "*.so",
    ".pytest_cache/",
    ".mypy_cache/",
    ".ruff_cache/",
    ".coverage",
    "htmlcov/",

    # Virtual environment
    ".venv/",
    "venv/",
    "env/",

    # Jupyter
    ".ipynb_checkpoints/",

    # Secrets
    ".env",
    ".env.*",
    "!.env.example",
    ".streamlit/secrets.toml",
    "*.pem",
    "*.key",

    # Private invoice data
    "InvoiceFlow-AI-Data/",
    "data/raw/",
    "data/interim/",
    "data/processed/",

    # Local databases and artifacts
    "*.db",
    "*.sqlite",
    "*.sqlite3",
    "models/",
    "artifacts/",
    "outputs/",
    "logs/",

    # IDE and operating system
    ".vscode/",
    ".idea/",
    ".DS_Store",
    "Thumbs.db",
]

if GITIGNORE_PATH.exists():
    existing_content = GITIGNORE_PATH.read_text(
        encoding="utf-8"
    )
else:
    existing_content = ""

existing_rules = {
    line.strip()
    for line in existing_content.splitlines()
    if line.strip() and not line.strip().startswith("#")
}

missing_rules = [
    rule
    for rule in REQUIRED_GITIGNORE_RULES
    if rule not in existing_rules
]

if missing_rules:
    block_lines = [
        "",
        "# InvoiceFlow AI — managed protection rules",
        *missing_rules,
        "",
    ]

    updated_content = (
        existing_content.rstrip()
        + "\n"
        + "\n".join(block_lines)
    )

    GITIGNORE_PATH.write_text(
        updated_content,
        encoding="utf-8",
    )

    print("Aturan baru yang ditambahkan:")

    for rule in missing_rules:
        print(f"  + {rule}")
else:
    print("ℹ️ Semua aturan .gitignore sudah tersedia.")

print(f"\n✅ .gitignore siap: {GITIGNORE_PATH}")

**Cell 11 — Code: Memasang PaddlePaddle dan dependensi**

In [ ]:
print("Memasang PaddlePaddle CPU baseline...")

run_command(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--disable-pip-version-check",
        "paddlepaddle==3.2.0",
        "-i",
        "https://www.paddlepaddle.org.cn/packages/stable/cpu/",
    ]
)

print("Memasang dependensi InvoiceFlow AI...")

run_command(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--disable-pip-version-check",
        "--upgrade-strategy",
        "only-if-needed",
        "-r",
        str(REQUIREMENTS_PATH),
    ]
)

print("✅ Instalasi dependensi selesai.")

**Cell 12 — Code: Memverifikasi import dan versi**

In [ ]:
from importlib.metadata import PackageNotFoundError, version

import cv2
import fitz
import numpy as np
import openpyxl
import paddle
import pandas as pd
import plotly
import sentence_transformers
import sklearn
import streamlit
import torch
from PIL import Image
from paddleocr import PaddleOCR


PACKAGE_DISTRIBUTIONS = [
    "paddlepaddle",
    "paddleocr",
    "PyMuPDF",
    "Pillow",
    "opencv-python-headless",
    "pandas",
    "numpy",
    "scikit-learn",
    "sentence-transformers",
    "streamlit",
    "plotly",
    "openpyxl",
    "pytest",
]

installed_versions = {}

for package_name in PACKAGE_DISTRIBUTIONS:
    try:
        installed_versions[package_name] = version(package_name)
    except PackageNotFoundError:
        installed_versions[package_name] = "NOT FOUND"

version_table = pd.DataFrame(
    [
        {
            "Package": package_name,
            "Version": package_version,
            "Status": (
                "READY"
                if package_version != "NOT FOUND"
                else "FAILED"
            ),
        }
        for package_name, package_version
        in installed_versions.items()
    ]
)

display(version_table)

missing_packages = [
    package_name
    for package_name, package_version
    in installed_versions.items()
    if package_version == "NOT FOUND"
]

if missing_packages:
    raise RuntimeError(
        f"Package belum tersedia: {missing_packages}"
    )

print("✅ Seluruh import dan package utama berhasil diverifikasi.")

**Cell 13 — Code: Membuat struktur data privat**

In [ ]:
RAW_DIR = DATA_ROOT / "raw"
INTERIM_DIR = DATA_ROOT / "interim"
PROCESSED_DIR = DATA_ROOT / "processed"
SAMPLES_DIR = DATA_ROOT / "samples"

DATA_DIRECTORIES = {
    "raw": RAW_DIR,
    "interim": INTERIM_DIR,
    "processed": PROCESSED_DIR,
    "samples": SAMPLES_DIR,
}

for directory in DATA_DIRECTORIES.values():
    directory.mkdir(parents=True, exist_ok=True)

resolved_repository = LOCAL_REPOSITORY_ROOT.resolve()
resolved_data_root = DATA_ROOT.resolve()

if resolved_data_root.is_relative_to(resolved_repository):
    raise RuntimeError(
        "Data privat tidak boleh berada di dalam repository GitHub."
    )

data_directory_table = pd.DataFrame(
    [
        {
            "Category": category,
            "Path": str(directory),
            "Purpose": {
                "raw": "Invoice asli; immutable",
                "interim": "Hasil preprocessing dan OCR sementara",
                "processed": "Data terstruktur dan tervalidasi",
                "samples": "Sampel kecil untuk eksperimen",
            }[category],
        }
        for category, directory in DATA_DIRECTORIES.items()
    ]
)

display(data_directory_table)

print("✅ Source code dan data privat telah dipisahkan.")

**Cell 14 — Code: Mengatur seed dan logging**

In [ ]:
import logging
import random


os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
paddle.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

logging.basicConfig(
    level=logging.INFO,
    format=(
        "%(asctime)s | %(levelname)s | "
        "%(name)s | %(message)s"
    ),
    force=True,
)

logger = logging.getLogger(PROJECT_NAME)

logger.info("Project: %s", PROJECT_NAME)
logger.info("Environment: %s", ENVIRONMENT_NAME)
logger.info("Run ID: %s", RUN_ID)
logger.info("Random seed: %d", RANDOM_SEED)

**Cell 15 — Code: Memeriksa hardware dan PaddlePaddle**

In [ ]:
nvidia_smi_path = shutil.which("nvidia-smi")
gpu_hardware_available = nvidia_smi_path is not None

if gpu_hardware_available:
    gpu_result = run_command(
        [
            nvidia_smi_path,
            "--query-gpu=name,memory.total",
            "--format=csv,noheader",
        ],
        check=False,
    )

    gpu_description = (
        gpu_result.stdout.strip()
        or "NVIDIA GPU terdeteksi"
    )
else:
    gpu_description = "Tidak terdeteksi"

# Tahap awal menggunakan CPU secara sengaja.
paddle.set_device("cpu")

print(f"GPU hardware   : {gpu_description}")
print(f"Torch CUDA     : {torch.cuda.is_available()}")
print(f"Paddle device  : {paddle.get_device()}")
print(f"Paddle version : {paddle.__version__}")

paddle.utils.run_check()

print("✅ Pemeriksaan PaddlePaddle berhasil.")

**Cell 16 — Code: Menguji akses baca dan tulis**

In [ ]:
storage_test_results = {}

test_locations = {
    "repository": LOCAL_REPOSITORY_ROOT,
    "private_data": DATA_ROOT,
}

for location_name, location_path in test_locations.items():
    test_file = location_path / f".write_test_{RUN_ID}.tmp"
    expected_content = f"InvoiceFlow AI — {RUN_ID}"

    try:
        test_file.write_text(
            expected_content,
            encoding="utf-8",
        )

        actual_content = test_file.read_text(
            encoding="utf-8",
        )

        storage_test_results[location_name] = (
            actual_content == expected_content
        )
    finally:
        test_file.unlink(missing_ok=True)

storage_test_table = pd.DataFrame(
    [
        {
            "Location": location_name,
            "Writable": test_passed,
            "Status": "READY" if test_passed else "FAILED",
        }
        for location_name, test_passed
        in storage_test_results.items()
    ]
)

display(storage_test_table)

if not all(storage_test_results.values()):
    raise OSError(
        "Satu atau lebih lokasi penyimpanan tidak dapat digunakan."
    )

print("✅ Pemeriksaan akses penyimpanan berhasil.")

**Cell 17 — Code: Memeriksa file sensitif**

In [ ]:
SENSITIVE_FILENAMES = {
    ".env",
    "secrets.toml",
    "credentials.json",
    "service-account.json",
}

sensitive_files_found = []

for file_path in LOCAL_REPOSITORY_ROOT.rglob("*"):
    if ".git" in file_path.parts:
        continue

    if (
        file_path.is_file()
        and file_path.name.lower() in SENSITIVE_FILENAMES
    ):
        sensitive_files_found.append(
            str(file_path.relative_to(LOCAL_REPOSITORY_ROOT))
        )

required_gitignore_rules = {
    ".env",
    ".streamlit/secrets.toml",
    "data/raw/",
    "data/interim/",
    "data/processed/",
}

gitignore_content = GITIGNORE_PATH.read_text(
    encoding="utf-8"
)

missing_gitignore_rules = sorted(
    rule
    for rule in required_gitignore_rules
    if rule not in gitignore_content
)

if sensitive_files_found:
    raise RuntimeError(
        "File sensitif ditemukan di repository: "
        f"{sensitive_files_found}"
    )

if missing_gitignore_rules:
    raise RuntimeError(
        "Aturan .gitignore belum lengkap: "
        f"{missing_gitignore_rules}"
    )

print("✅ Tidak ditemukan file kredensial di repository.")
print("✅ Aturan perlindungan data tersedia di .gitignore.")

**Cell 18 — Code: Menilai kesiapan environment**

In [ ]:
from packaging.version import Version


readiness_checks = {
    "Supported Python": (
        SUPPORTED_PYTHON_MIN
        <= CURRENT_PYTHON
        <= SUPPORTED_PYTHON_MAX
    ),
    "64-bit runtime": IS_64_BIT,
    "Google Colab": IN_COLAB,
    "Google Drive mounted": DRIVE_MOUNT_POINT.exists(),
    "Git repository connected": git_directory.exists(),
    "requirements.txt available": REQUIREMENTS_PATH.exists(),
    ".gitignore available": GITIGNORE_PATH.exists(),
    "All dependencies imported": not missing_packages,
    "PaddlePaddle >= 3.0": (
        Version(paddle.__version__) >= Version("3.0.0")
    ),
    "Code directories available": all(
        directory.exists()
        for directory in CODE_DIRECTORIES
    ),
    "Data directories available": all(
        directory.exists()
        for directory in DATA_DIRECTORIES.values()
    ),
    "Storage writable": all(
        storage_test_results.values()
    ),
    "No sensitive files": not sensitive_files_found,
}

readiness_table = pd.DataFrame(
    [
        {
            "Check": check_name,
            "Result": check_result,
            "Status": (
                "READY"
                if check_result
                else "FAILED"
            ),
        }
        for check_name, check_result
        in readiness_checks.items()
    ]
)

display(readiness_table)

failed_checks = [
    check_name
    for check_name, check_result
    in readiness_checks.items()
    if not check_result
]

if failed_checks:
    raise RuntimeError(
        f"Environment belum siap: {failed_checks}"
    )

print("✅ ENVIRONMENT STATUS: READY")

**Cell 19 — Code: Menyimpan environment manifest**

In [ ]:
import hashlib
import json


git_commit_result = run_command(
    ["git", "rev-parse", "HEAD"],
    working_directory=LOCAL_REPOSITORY_ROOT,
    check=False,
)

git_commit = (
    git_commit_result.stdout.strip()
    or "NO_COMMIT_YET"
)

requirements_hash = hashlib.sha256(
    REQUIREMENTS_PATH.read_bytes()
).hexdigest()

environment_manifest = {
    "project": {
        "name": PROJECT_NAME,
        "version": PROJECT_VERSION,
        "environment": ENVIRONMENT_NAME,
        "run_id": RUN_ID,
    },
    "runtime": runtime_information,
    "repository": {
        "owner": GITHUB_OWNER,
        "name": GITHUB_REPOSITORY,
        "branch": DEFAULT_BRANCH,
        "commit": git_commit,
        "local_root": str(LOCAL_REPOSITORY_ROOT),
    },
    "storage": {
        "data_root": str(DATA_ROOT),
        "directories": {
            name: str(path)
            for name, path in DATA_DIRECTORIES.items()
        },
    },
    "reproducibility": {
        "random_seed": RANDOM_SEED,
        "requirements_sha256": requirements_hash,
    },
    "hardware": {
        "gpu_hardware_available": gpu_hardware_available,
        "gpu_description": gpu_description,
        "paddle_device": paddle.get_device(),
        "torch_cuda_available": torch.cuda.is_available(),
    },
    "packages": installed_versions,
    "readiness_checks": readiness_checks,
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
}

manifest_path = DATA_ROOT / "environment_manifest.json"
temporary_manifest_path = DATA_ROOT / "environment_manifest.tmp"

temporary_manifest_path.write_text(
    json.dumps(
        environment_manifest,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

temporary_manifest_path.replace(manifest_path)

print(f"✅ Environment manifest: {manifest_path}")

**Cell 20 — Code: Menampilkan status akhir**

In [ ]:
git_status_result = run_command(
    ["git", "status", "--short"],
    working_directory=LOCAL_REPOSITORY_ROOT,
    check=False,
)

git_changes = (
    git_status_result.stdout.strip()
    or "Tidak ada perubahan lokal"
)

print("=" * 70)
print("INVOICEFLOW AI — ENVIRONMENT INITIALIZATION COMPLETE")
print("=" * 70)
print(f"Status       : READY")
print(f"Project      : {PROJECT_NAME}")
print(f"Run ID       : {RUN_ID}")
print(f"Repository   : {GITHUB_OWNER}/{GITHUB_REPOSITORY}")
print(f"Code root    : {LOCAL_REPOSITORY_ROOT}")
print(f"Data root    : {DATA_ROOT}")
print(f"Paddle       : {paddle.__version__}")
print(f"Device       : {paddle.get_device()}")
print(f"Manifest     : {manifest_path}")
print("\nGit working tree:")
print(git_changes)
print("=" * 70)

**Cell 21 — Markdown: Findings dan langkah berikutnya**

## 10. Findings

Environment dinyatakan siap apabila Cell 20 menampilkan:

```text
Status : READY
```

Setelah environment siap, lanjutkan ke notebook `01_data_collection_and_audit.ipynb`.

Data privat dan artifact hasil pemrosesan tetap disimpan di Google Drive dan tidak dimasukkan ke repository GitHub.
